# Curriculum Learning Analysis - Monopoly RL Training

This notebook analyzes training progress with curriculum learning features:
- **Phase 1** (Games 0-999): 15 rounds, 10× players max actions
- **Phase 2** (Games 1000-2999): 25 rounds, 20× players max actions
- **Phase 3** (Games 3000+): 50 rounds, 50× players max actions

**Metrics analyzed:**
- Performance by curriculum phase
- Action usage patterns across phases
- Game length and completion statistics
- Learning progression through curriculum
- Phase transition effects

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
import pandas as pd
from datetime import datetime

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# Define curriculum phases
PHASE_1_END = 1000
PHASE_2_END = 3000

def get_phase(game_num):
    """Get curriculum phase for a game number."""
    if game_num < PHASE_1_END:
        return 1
    elif game_num < PHASE_2_END:
        return 2
    else:
        return 3

def get_max_rounds(phase):
    """Get max rounds for a phase."""
    return {1: 15, 2: 25, 3: 50}[phase]

def get_max_actions_multiplier(phase):
    """Get action multiplier for a phase."""
    return {1: 10, 2: 20, 3: 50}[phase]

print("Curriculum Learning Analysis Notebook Loaded")
print(f"Phase boundaries: {PHASE_1_END}, {PHASE_2_END}")

## 1. Load Training Data

In [ ]:
# Configuration
CHECKPOINT_DIR = Path('models/checkpoints')
LOG_DIR = Path('logs')
RECORDING_DIR = Path('recordings')

# Find latest history file
history_files = sorted(CHECKPOINT_DIR.glob('history_*.json'))

if not history_files:
    print("❌ No history files found. Make sure training has run and saved checkpoints.")
    print(f"Looking in: {CHECKPOINT_DIR.absolute()}")
else:
    latest_history = history_files[-1]
    print(f"✓ Loading: {latest_history.name}")
    
    with open(latest_history) as f:
        history = json.load(f)
    
    agent_ids = history['agent_ids']
    games_played = history['games_played']
    wins = history['wins']
    game_history = history['game_history']
    game_rewards = history['game_rewards']
    max_rewards = history['max_rewards']
    
    total_games = len(game_history)
    current_phase = get_phase(total_games - 1)
    
    print(f"\n📊 Training Summary:")
    print(f"   Total games: {total_games}")
    print(f"   Current phase: {current_phase}")
    print(f"   Agents: {', '.join(agent_ids)}")
    
    # Calculate phase distribution
    phase_counts = {1: 0, 2: 0, 3: 0}
    for i in range(total_games):
        phase_counts[get_phase(i)] += 1
    
    print(f"\n📈 Games by Phase:")
    print(f"   Phase 1: {phase_counts[1]} games (0-{PHASE_1_END-1})")
    print(f"   Phase 2: {phase_counts[2]} games ({PHASE_1_END}-{PHASE_2_END-1})")
    print(f"   Phase 3: {phase_counts[3]} games ({PHASE_2_END}+)")

## 2. Curriculum Phase Overview

In [ ]:
# Create phase-annotated timeline
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Prepare phase markers
game_numbers = list(range(total_games))
phases = [get_phase(i) for i in game_numbers]

# Define phase colors
phase_colors = {1: '#ff9999', 2: '#99ccff', 3: '#99ff99'}
phase_labels = {
    1: 'Phase 1: Basic (15 rds, 10×players actions)',
    2: 'Phase 2: Strategic (25 rds, 20×players actions)',
    3: 'Phase 3: Full (50 rds, 50×players actions)'
}

# Plot 1: Win rates by phase with phase backgrounds
ax1 = axes[0]
for agent_id in agent_ids:
    # Calculate rolling win rate
    wins_list = []
    games_list = []
    win_rate = []
    
    for i, game in enumerate(game_history):
        if agent_id in game['participants']:
            games_list.append(i)
            wins_list.append(1 if game['winner'] == agent_id else 0)
            # Rolling average over last 50 games for this agent
            window = min(50, len(wins_list))
            win_rate.append(np.mean(wins_list[-window:]))
    
    ax1.plot(games_list, win_rate, label=agent_id, alpha=0.7, linewidth=1.5)

# Add phase backgrounds
ax1.axvspan(0, PHASE_1_END, alpha=0.1, color=phase_colors[1], label='Phase 1')
ax1.axvspan(PHASE_1_END, min(PHASE_2_END, total_games), alpha=0.1, color=phase_colors[2], label='Phase 2')
if total_games > PHASE_2_END:
    ax1.axvspan(PHASE_2_END, total_games, alpha=0.1, color=phase_colors[3], label='Phase 3')

# Add phase transition lines
if total_games > PHASE_1_END:
    ax1.axvline(PHASE_1_END, color='red', linestyle='--', alpha=0.5, linewidth=2)
if total_games > PHASE_2_END:
    ax1.axvline(PHASE_2_END, color='red', linestyle='--', alpha=0.5, linewidth=2)

ax1.set_xlabel('Game Number')
ax1.set_ylabel('Win Rate (50-game rolling avg)')
ax1.set_title('Win Rates Across Curriculum Phases')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 1])

# Plot 2: Average rewards by phase
ax2 = axes[1]
for agent_id in agent_ids:
    rewards = [r if r is not None else np.nan for r in game_rewards[agent_id]]
    rolling_avg = pd.Series(rewards).rolling(window=50, min_periods=1).mean()
    ax2.plot(rolling_avg, label=agent_id, alpha=0.7, linewidth=1.5)

# Add phase backgrounds
ax2.axvspan(0, PHASE_1_END, alpha=0.1, color=phase_colors[1])
ax2.axvspan(PHASE_1_END, min(PHASE_2_END, total_games), alpha=0.1, color=phase_colors[2])
if total_games > PHASE_2_END:
    ax2.axvspan(PHASE_2_END, total_games, alpha=0.1, color=phase_colors[3])

# Add phase transition lines
if total_games > PHASE_1_END:
    ax2.axvline(PHASE_1_END, color='red', linestyle='--', alpha=0.5, linewidth=2)
if total_games > PHASE_2_END:
    ax2.axvline(PHASE_2_END, color='red', linestyle='--', alpha=0.5, linewidth=2)

ax2.set_xlabel('Game Number')
ax2.set_ylabel('Reward (50-game rolling avg)')
ax2.set_title('Rewards Across Curriculum Phases')
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax2.grid(True, alpha=0.3)

# Plot 3: Number of players per game
ax3 = axes[2]
players_per_game = [len(game['participants']) for game in game_history]
rolling_players = pd.Series(players_per_game).rolling(window=50, min_periods=1).mean()
ax3.plot(rolling_players, color='darkblue', linewidth=2, label='Avg players (50-game rolling)')
ax3.scatter(range(total_games), players_per_game, alpha=0.3, s=10, color='lightblue', label='Actual')

# Add phase backgrounds
ax3.axvspan(0, PHASE_1_END, alpha=0.1, color=phase_colors[1])
ax3.axvspan(PHASE_1_END, min(PHASE_2_END, total_games), alpha=0.1, color=phase_colors[2])
if total_games > PHASE_2_END:
    ax3.axvspan(PHASE_2_END, total_games, alpha=0.1, color=phase_colors[3])

# Add phase transition lines
if total_games > PHASE_1_END:
    ax3.axvline(PHASE_1_END, color='red', linestyle='--', alpha=0.5, linewidth=2, label='Phase transition')
if total_games > PHASE_2_END:
    ax3.axvline(PHASE_2_END, color='red', linestyle='--', alpha=0.5, linewidth=2)

ax3.set_xlabel('Game Number')
ax3.set_ylabel('Number of Players')
ax3.set_title('Player Count Distribution Across Training')
ax3.legend(loc='upper right', fontsize=8)
ax3.grid(True, alpha=0.3)
ax3.set_ylim([3.5, 6.5])

plt.tight_layout()
plt.show()

print("\n📊 Player distribution:")
print(f"   Average: {np.mean(players_per_game):.2f} players/game")
print(f"   Range: {min(players_per_game)} - {max(players_per_game)} players")

## 3. Performance Statistics by Phase

In [ ]:
# Calculate statistics for each phase
phase_stats = {agent_id: {1: [], 2: [], 3: []} for agent_id in agent_ids}
phase_wins = {agent_id: {1: 0, 2: 0, 3: 0} for agent_id in agent_ids}
phase_games = {agent_id: {1: 0, 2: 0, 3: 0} for agent_id in agent_ids}

for i, game in enumerate(game_history):
    phase = get_phase(i)
    winner = game['winner']
    
    for agent_id in game['participants']:
        phase_games[agent_id][phase] += 1
        phase_stats[agent_id][phase].append(game['rewards'][agent_id])
        if agent_id == winner:
            phase_wins[agent_id][phase] += 1

# Create comparison table
print("\n" + "="*100)
print("PERFORMANCE BY CURRICULUM PHASE")
print("="*100)

for phase in [1, 2, 3]:
    max_rds = get_max_rounds(phase)
    max_acts = get_max_actions_multiplier(phase)
    print(f"\n{'='*100}")
    print(f"PHASE {phase}: {max_rds} rounds, {max_acts}× players max actions")
    print(f"{'='*100}")
    print(f"{'Agent':<12} {'Games':<8} {'Wins':<8} {'Win Rate':<12} {'Avg Reward':<15} {'Std Reward':<15}")
    print("-"*100)
    
    # Sort by win rate for this phase
    agent_phase_data = []
    for agent_id in agent_ids:
        games = phase_games[agent_id][phase]
        wins = phase_wins[agent_id][phase]
        wr = wins / games if games > 0 else 0
        rewards = phase_stats[agent_id][phase]
        avg_reward = np.mean(rewards) if rewards else 0
        std_reward = np.std(rewards) if rewards else 0
        agent_phase_data.append((agent_id, games, wins, wr, avg_reward, std_reward))
    
    agent_phase_data.sort(key=lambda x: x[3], reverse=True)
    
    for agent_id, games, wins, wr, avg_reward, std_reward in agent_phase_data:
        if games > 0:
            print(f"{agent_id:<12} {games:<8} {wins:<8} {wr:>10.3f}  {avg_reward:>12.2f}  {std_reward:>12.2f}")

print("\n" + "="*100)

## 4. Phase-to-Phase Performance Comparison

In [ ]:
# Create visualizations comparing performance across phases
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Win rates by phase
ax1 = axes[0, 0]
phases_to_plot = [1, 2, 3]
x = np.arange(len(agent_ids))
width = 0.25

for i, phase in enumerate(phases_to_plot):
    win_rates = [phase_wins[a][phase] / phase_games[a][phase] if phase_games[a][phase] > 0 else 0 
                 for a in agent_ids]
    ax1.bar(x + i*width, win_rates, width, label=f'Phase {phase}', alpha=0.8)

ax1.set_xlabel('Agent')
ax1.set_ylabel('Win Rate')
ax1.set_title('Win Rates by Curriculum Phase')
ax1.set_xticks(x + width)
ax1.set_xticklabels(agent_ids, rotation=45, ha='right')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_ylim([0, 1])

# Plot 2: Average rewards by phase
ax2 = axes[0, 1]
for i, phase in enumerate(phases_to_plot):
    avg_rewards = [np.mean(phase_stats[a][phase]) if phase_stats[a][phase] else 0 
                   for a in agent_ids]
    ax2.bar(x + i*width, avg_rewards, width, label=f'Phase {phase}', alpha=0.8)

ax2.set_xlabel('Agent')
ax2.set_ylabel('Average Reward')
ax2.set_title('Average Rewards by Curriculum Phase')
ax2.set_xticks(x + width)
ax2.set_xticklabels(agent_ids, rotation=45, ha='right')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# Plot 3: Games played by phase
ax3 = axes[1, 0]
for i, phase in enumerate(phases_to_plot):
    games_counts = [phase_games[a][phase] for a in agent_ids]
    ax3.bar(x + i*width, games_counts, width, label=f'Phase {phase}', alpha=0.8)

ax3.set_xlabel('Agent')
ax3.set_ylabel('Games Played')
ax3.set_title('Games Played by Curriculum Phase')
ax3.set_xticks(x + width)
ax3.set_xticklabels(agent_ids, rotation=45, ha='right')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# Plot 4: Reward standard deviation by phase
ax4 = axes[1, 1]
for i, phase in enumerate(phases_to_plot):
    std_rewards = [np.std(phase_stats[a][phase]) if len(phase_stats[a][phase]) > 1 else 0 
                   for a in agent_ids]
    ax4.bar(x + i*width, std_rewards, width, label=f'Phase {phase}', alpha=0.8)

ax4.set_xlabel('Agent')
ax4.set_ylabel('Reward Std Dev')
ax4.set_title('Reward Variability by Curriculum Phase')
ax4.set_xticks(x + width)
ax4.set_xticklabels(agent_ids, rotation=45, ha='right')
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Analyzing Phase Transitions

How do agents perform immediately after phase transitions?

In [ ]:
# Analyze performance around phase transitions
transition_window = 100  # Look at +/- 100 games around transition

if total_games > PHASE_1_END + transition_window:
    print("\n" + "="*80)
    print(f"PHASE TRANSITION ANALYSIS (±{transition_window} games around transitions)")
    print("="*80)
    
    for transition_point, trans_name in [(PHASE_1_END, "Phase 1→2"), (PHASE_2_END, "Phase 2→3")]:
        if total_games <= transition_point:
            continue
        
        print(f"\n{trans_name} (at game {transition_point}):")
        print("-"*80)
        
        # Calculate metrics before and after
        before_start = max(0, transition_point - transition_window)
        before_end = transition_point
        after_start = transition_point
        after_end = min(total_games, transition_point + transition_window)
        
        before_games = game_history[before_start:before_end]
        after_games = game_history[after_start:after_end]
        
        print(f"{'Agent':<12} {'Before WR':<12} {'After WR':<12} {'Change':<12} {'Before Rew':<15} {'After Rew':<15}")
        print("-"*80)
        
        for agent_id in agent_ids:
            # Before transition
            before_wins = sum(1 for g in before_games if agent_id in g['participants'] and g['winner'] == agent_id)
            before_count = sum(1 for g in before_games if agent_id in g['participants'])
            before_wr = before_wins / before_count if before_count > 0 else 0
            before_rewards = [g['rewards'][agent_id] for g in before_games if agent_id in g['participants']]
            before_avg_rew = np.mean(before_rewards) if before_rewards else 0
            
            # After transition
            after_wins = sum(1 for g in after_games if agent_id in g['participants'] and g['winner'] == agent_id)
            after_count = sum(1 for g in after_games if agent_id in g['participants'])
            after_wr = after_wins / after_count if after_count > 0 else 0
            after_rewards = [g['rewards'][agent_id] for g in after_games if agent_id in g['participants']]
            after_avg_rew = np.mean(after_rewards) if after_rewards else 0
            
            wr_change = after_wr - before_wr
            change_symbol = "↑" if wr_change > 0.01 else "↓" if wr_change < -0.01 else "→"
            
            if before_count > 0 and after_count > 0:
                print(f"{agent_id:<12} {before_wr:>10.3f}  {after_wr:>10.3f}  {change_symbol} {wr_change:>8.3f}  "
                      f"{before_avg_rew:>12.2f}  {after_avg_rew:>12.2f}")
else:
    print("\n⚠ Not enough games to analyze phase transitions yet.")

## 6. Curriculum Learning Impact

Visualize learning velocity and improvement across phases

In [ ]:
# Calculate learning curves with phase annotations
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Plot 1: Cumulative improvement in win rate
ax1 = axes[0]
for agent_id in agent_ids:
    # Get all games for this agent
    agent_games = [(i, game) for i, game in enumerate(game_history) if agent_id in game['participants']]
    
    if len(agent_games) < 20:
        continue
    
    # Calculate win rate improvement (current - baseline)
    baseline_wr = np.mean([1 if g[1]['winner'] == agent_id else 0 
                           for g in agent_games[:min(50, len(agent_games))]])
    
    game_nums = [g[0] for g in agent_games]
    win_rates = []
    
    for i in range(len(agent_games)):
        window_start = max(0, i - 25)
        window_end = min(len(agent_games), i + 25)
        window_games = agent_games[window_start:window_end]
        wr = np.mean([1 if g[1]['winner'] == agent_id else 0 for g in window_games])
        win_rates.append(wr - baseline_wr)
    
    ax1.plot(game_nums, win_rates, label=agent_id, alpha=0.7, linewidth=1.5)

# Add phase backgrounds
ax1.axvspan(0, PHASE_1_END, alpha=0.1, color=phase_colors[1])
ax1.axvspan(PHASE_1_END, min(PHASE_2_END, total_games), alpha=0.1, color=phase_colors[2])
if total_games > PHASE_2_END:
    ax1.axvspan(PHASE_2_END, total_games, alpha=0.1, color=phase_colors[3])

if total_games > PHASE_1_END:
    ax1.axvline(PHASE_1_END, color='red', linestyle='--', alpha=0.5, linewidth=2)
if total_games > PHASE_2_END:
    ax1.axvline(PHASE_2_END, color='red', linestyle='--', alpha=0.5, linewidth=2)

ax1.axhline(0, color='black', linestyle='-', alpha=0.3, linewidth=1)
ax1.set_xlabel('Game Number')
ax1.set_ylabel('Win Rate Change from Baseline')
ax1.set_title('Win Rate Improvement Over Training (50-game window)')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)

# Plot 2: Reward improvement
ax2 = axes[1]
for agent_id in agent_ids:
    rewards = [r if r is not None else np.nan for r in game_rewards[agent_id]]
    
    # Calculate baseline (first 50 games)
    valid_rewards = [r for r in rewards[:50] if not np.isnan(r)]
    if not valid_rewards:
        continue
    baseline_reward = np.mean(valid_rewards)
    
    # Calculate improvement
    rolling_reward = pd.Series(rewards).rolling(window=50, min_periods=1).mean()
    improvement = rolling_reward - baseline_reward
    
    ax2.plot(improvement, label=agent_id, alpha=0.7, linewidth=1.5)

# Add phase backgrounds
ax2.axvspan(0, PHASE_1_END, alpha=0.1, color=phase_colors[1])
ax2.axvspan(PHASE_1_END, min(PHASE_2_END, total_games), alpha=0.1, color=phase_colors[2])
if total_games > PHASE_2_END:
    ax2.axvspan(PHASE_2_END, total_games, alpha=0.1, color=phase_colors[3])

if total_games > PHASE_1_END:
    ax2.axvline(PHASE_1_END, color='red', linestyle='--', alpha=0.5, linewidth=2)
if total_games > PHASE_2_END:
    ax2.axvline(PHASE_2_END, color='red', linestyle='--', alpha=0.5, linewidth=2)

ax2.axhline(0, color='black', linestyle='-', alpha=0.3, linewidth=1)
ax2.set_xlabel('Game Number')
ax2.set_ylabel('Reward Change from Baseline')
ax2.set_title('Reward Improvement Over Training (50-game rolling average)')
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Training Summary & Recommendations

In [ ]:
print("\n" + "="*80)
print("CURRICULUM LEARNING SUMMARY")
print("="*80)

print(f"\n📊 Overall Progress:")
print(f"   Total games completed: {total_games}")
print(f"   Current phase: {current_phase}")
print(f"   Games remaining in phase: ", end="")
if current_phase == 1:
    print(f"{PHASE_1_END - total_games} until Phase 2")
elif current_phase == 2:
    print(f"{PHASE_2_END - total_games} until Phase 3")
else:
    print("In final phase")

# Calculate overall trends
print(f"\n📈 Top Performers by Phase:")
for phase in [1, 2, 3]:
    if phase_counts[phase] == 0:
        continue
    
    # Find top 3 agents
    agent_wrs = [(a, phase_wins[a][phase] / phase_games[a][phase] if phase_games[a][phase] > 0 else 0) 
                 for a in agent_ids]
    agent_wrs.sort(key=lambda x: x[1], reverse=True)
    
    max_rds = get_max_rounds(phase)
    max_acts = get_max_actions_multiplier(phase)
    print(f"\n   Phase {phase} ({max_rds} rds, {max_acts}× actions):")
    for i, (agent_id, wr) in enumerate(agent_wrs[:3], 1):
        games = phase_games[agent_id][phase]
        if games > 0:
            print(f"      {i}. {agent_id}: {wr:.3f} win rate ({phase_wins[agent_id][phase]}/{games} games)")

# Check for learning progress
print(f"\n🎯 Learning Indicators:")
improving_agents = []
for agent_id in agent_ids:
    # Compare Phase 1 to current phase
    phase1_wr = phase_wins[agent_id][1] / phase_games[agent_id][1] if phase_games[agent_id][1] > 0 else 0
    current_wr = phase_wins[agent_id][current_phase] / phase_games[agent_id][current_phase] \
                 if phase_games[agent_id][current_phase] > 0 else 0
    
    if current_wr > phase1_wr + 0.05:  # 5% improvement threshold
        improving_agents.append((agent_id, current_wr - phase1_wr))

if improving_agents:
    improving_agents.sort(key=lambda x: x[1], reverse=True)
    print(f"   Agents showing improvement from Phase 1:")
    for agent_id, improvement in improving_agents:
        print(f"      • {agent_id}: +{improvement:.3f} win rate")
else:
    print(f"   ⚠ No significant improvements detected yet")

# Recommendations
print(f"\n💡 Recommendations:")
if total_games < 500:
    print(f"   • Early training - continue to build baseline performance")
    print(f"   • Monitor for basic competency before phase transitions")
elif total_games < PHASE_1_END:
    print(f"   • In Phase 1 - focus on basic strategy learning")
    print(f"   • {PHASE_1_END - total_games} games until Phase 2 transition")
elif total_games < PHASE_2_END:
    print(f"   • In Phase 2 - developing strategic play")
    print(f"   • {PHASE_2_END - total_games} games until Phase 3 transition")
else:
    print(f"   • In Phase 3 - full game complexity")
    print(f"   • Continue training for advanced strategy refinement")

# Check win rate distribution
current_phase_wrs = [phase_wins[a][current_phase] / phase_games[a][current_phase] 
                     if phase_games[a][current_phase] > 0 else 0 
                     for a in agent_ids]
wr_std = np.std(current_phase_wrs)
if wr_std < 0.05:
    print(f"   ⚠ Low win rate variance ({wr_std:.3f}) - agents performing similarly")
    print(f"      Consider adjusting learning rates or exploration")

print("\n" + "="*80)

## 8. Export Phase Statistics

In [ ]:
# Create comprehensive DataFrame with phase statistics
export_data = []

for agent_id in agent_ids:
    for phase in [1, 2, 3]:
        if phase_games[agent_id][phase] > 0:
            rewards = phase_stats[agent_id][phase]
            export_data.append({
                'Agent': agent_id,
                'Phase': phase,
                'Max_Rounds': get_max_rounds(phase),
                'Max_Actions_Multiplier': get_max_actions_multiplier(phase),
                'Games_Played': phase_games[agent_id][phase],
                'Wins': phase_wins[agent_id][phase],
                'Win_Rate': phase_wins[agent_id][phase] / phase_games[agent_id][phase],
                'Avg_Reward': np.mean(rewards),
                'Std_Reward': np.std(rewards),
                'Min_Reward': np.min(rewards),
                'Max_Reward': np.max(rewards)
            })

df_curriculum = pd.DataFrame(export_data)
df_curriculum = df_curriculum.sort_values(['Phase', 'Win_Rate'], ascending=[True, False])

# Save to CSV
output_file = f'curriculum_results_game_{total_games}.csv'
df_curriculum.to_csv(output_file, index=False)
print(f"✓ Curriculum results exported to {output_file}")

# Display summary table
print(f"\nCurriculum Statistics by Phase:")
print(df_curriculum.to_string(index=False))

# Save timestamp
timestamp_file = 'last_analysis_timestamp.txt'
with open(timestamp_file, 'w') as f:
    f.write(f"Last analysis: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Total games: {total_games}\n")
    f.write(f"Current phase: {current_phase}\n")

print(f"\n✓ Analysis complete! Timestamp saved to {timestamp_file}")